<a href="https://colab.research.google.com/github/Clint9991/Global-Fleet-PySpark-ETL/blob/main/Global_Fleet_ETL_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install PySpark
!pip install pyspark

In [2]:
# 2. Import and start the Spark Session
from pyspark.sql import SparkSession

print("Starting the Big Data Engine...")
spark = SparkSession.builder \
    .appName("Global_Logistics_ETL") \
    .getOrCreate()

print("Spark Session Successfully Created! 🚀")
print(f"Spark Version: {spark.version}")

Starting the Big Data Engine...
Spark Session Successfully Created! 🚀
Spark Version: 4.0.2


In [4]:
from pyspark.sql.functions import rand, randn, expr, current_timestamp, round

print("Generating 1,000,000 rows of Global Fleet Telematics Data...")

# 1. Create a base dataframe with 1 million rows
df_raw = spark.range(0, 1000000)

# 2. Adding synthetic logistics columns natively in Spark (Blazing Fast)
df_telematics = df_raw.select(
    (expr("id % 5000 + 1000")).cast("int").alias("truck_id"), # 5000 unique trucks
    (current_timestamp() - expr("INTERVAL 1 SECOND") * rand() * 86400 * 30).alias("gps_ping_time"), # Random time in last 30 days
    round(rand() * 120, 2).alias("speed_kmh"), # Speed between 0 and 120 km/h
    round(randn() * 15 + 90, 2).alias("engine_temp_c"), # Normal distribution around 90C
    round(rand() * 5 + 2, 2).alias("fuel_efficiency_kml"), # Fuel efficiency
    round(rand() * 100, 2).alias("route_risk_score") # Risk score 0-100
)

# 3. Show the first 5 rows and count the total
df_telematics.show(5)
print(f"Total Rows Generated: {df_telematics.count():,}")

Generating 1,000,000 rows of Global Fleet Telematics Data...
+--------+--------------------+---------+-------------+-------------------+----------------+
|truck_id|       gps_ping_time|speed_kmh|engine_temp_c|fuel_efficiency_kml|route_risk_score|
+--------+--------------------+---------+-------------+-------------------+----------------+
|    1000|2026-02-07 11:22:...|    80.93|        87.45|               5.82|           45.44|
|    1001|2026-02-21 03:55:...|    19.25|        93.25|               5.32|           91.95|
|    1002|2026-01-25 19:17:...|   100.19|        80.01|                2.2|            0.82|
|    1003|2026-02-08 09:50:...|   111.93|        85.66|               6.49|            9.02|
|    1004|2026-02-19 21:14:...|    67.21|        85.72|               5.54|            20.0|
+--------+--------------------+---------+-------------+-------------------+----------------+
only showing top 5 rows
Total Rows Generated: 1,000,000


In [5]:
from pyspark.sql.functions import when, lit

print("Injecting real-world anomalies into the dataset...")

# Injecting missing values, negative speeds, and extreme temperatures
df_dirty = df_telematics.withColumn(
    "gps_ping_time",
    when(rand() < 0.05, lit(None)).otherwise(df_telematics.gps_ping_time) # 5% missing GPS
).withColumn(
    "speed_kmh",
    when(rand() < 0.02, df_telematics.speed_kmh * -1).otherwise(df_telematics.speed_kmh) # 2% negative speeds
).withColumn(
    "engine_temp_c",
    when(rand() < 0.01, lit(250.0)).otherwise(df_telematics.engine_temp_c) # 1% sensor malfunctions (250C)
)

print("Anomalies Injected! Here is the summary showing the errors:")
# Using summary() to prove the errors exist
df_dirty.select("gps_ping_time", "speed_kmh", "engine_temp_c").summary("count", "min", "max").show()

Injecting real-world anomalies into the dataset...
Anomalies Injected! Here is the summary showing the errors:
+-------+---------+-------------+
|summary|speed_kmh|engine_temp_c|
+-------+---------+-------------+
|  count|  1000000|      1000000|
|    min|  -119.98|        19.37|
|    max|    120.0|        250.0|
+-------+---------+-------------+



In [6]:
from pyspark.sql.functions import col, abs, when

print("Initiating Enterprise Data Cleaning Pipeline...")

# 1. Drop rows with missing GPS data
df_clean = df_dirty.dropna(subset=["gps_ping_time"])

# 2. Fixing negative speeds and remove impossible engine temperatures
df_clean = df_clean.withColumn("speed_kmh", abs(col("speed_kmh"))) \
                   .filter(col("engine_temp_c") <= 150.0)

# 3. Feature Engineering: Creating a Business-Ready Risk Category
df_clean = df_clean.withColumn(
    "risk_category",
    when(col("route_risk_score") >= 80, "High Risk")
    .when(col("route_risk_score") >= 40, "Medium Risk")
    .otherwise("Low Risk")
)

print("Pipeline Execution Complete! Verifying clean data...")

# Showing the new clean summary and the final row count
df_clean.select("gps_ping_time", "speed_kmh", "engine_temp_c").summary("count", "min", "max").show()
print(f"Final Cleaned Row Count: {df_clean.count():,}")

Initiating Enterprise Data Cleaning Pipeline...
Pipeline Execution Complete! Verifying clean data...
+-------+---------+-------------+
|summary|speed_kmh|engine_temp_c|
+-------+---------+-------------+
|  count|   940272|       940272|
|    min|      0.0|        19.37|
|    max|    120.0|       149.91|
+-------+---------+-------------+

Final Cleaned Row Count: 940,272


In [8]:
import os

print("Initiating Load Phase: Saving to Parquet...")

# Defining the output path
output_path = "/content/cleaned_telematics_data.parquet"

# Writing the data to Parquet, partitioned by our new business logic
df_clean.write.mode("overwrite").partitionBy("risk_category").parquet(output_path)

print(f"Success! Data loaded into partitioned Parquet format at: {output_path}")

# Verifying the folders were created
print("\nVerifying Partitioned Folders:")
for folder in os.listdir(output_path):
    print(f" - {folder}")

Initiating Load Phase: Saving to Parquet...
Success! Data loaded into partitioned Parquet format at: /content/cleaned_telematics_data.parquet

Verifying Partitioned Folders:
 - risk_category=Low Risk
 - _SUCCESS
 - risk_category=High Risk
 - risk_category=Medium Risk
 - ._SUCCESS.crc
